In [0]:
# ============================================================
# 1. SETUP E CREDENCIAIS (ecommerce_categorias)
# ============================================================
import os
from dotenv import load_dotenv
from pyspark.sql.functions import current_timestamp, year, month, col, count, date_format

# Carrega o .env
load_dotenv("../env")
client_id = os.getenv("CLIENT_ID")
tenant_id = os.getenv("TENANT_ID")
client_secret = os.getenv("CLIENT_SECRET")
storage_account = os.getenv("STORAGE_ACCOUNT_NAME")

# Opções de autenticação injetadas
adls_options = {
    "fs.azure.account.auth.type": "OAuth",
    "fs.azure.account.oauth.provider.type": "org.apache.hadoop.fs.azurebfs.oauth2.ClientCredsTokenProvider",
    "fs.azure.account.oauth2.client.id": client_id,
    "fs.azure.account.oauth2.client.secret": client_secret,
    "fs.azure.account.oauth2.client.endpoint": f"https://login.microsoftonline.com/{tenant_id}/oauth2/token"
}

tabela = "ecommerce_categorias"
path_raw = f"abfss://raw@{storage_account}.dfs.core.windows.net/batch-data/{tabela}.csv"
path_bronze = f"abfss://squad3@{storage_account}.dfs.core.windows.net/bronze/{tabela}"

print(f"Configuração finalizada para a tabela: {tabela}")

In [0]:
# ============================================================
# 2. EXTRAÇÃO E CARGA (RAW -> BRONZE)
# ============================================================
print(f"Lendo {tabela} da camada Raw...")

# 1. LEITURA DA RAW
print(f"Lendo dados da Raw com inferSchema=false...")
df_raw = (
    spark.read
    .format("csv")
    .option("header", "true")
    .option("inferSchema", "false") # Mantendo tudo como String
    .options(**adls_options) 
    .load(path_raw)
)

print(f"Total de registros encontrados: {df_raw.count()}")

# 2. ADICIONANDO AS COLUNAS DE PARTIÇÃO E AUDITORIA
df_bronze = (
    df_raw
    .withColumn("bronze_ingested_at", current_timestamp())
    .withColumn("bronze_source_file", col("_metadata.file_path"))
    .withColumn("ano_particao", year(col("bronze_ingested_at")))
    .withColumn("mes_particao", month(col("bronze_ingested_at")))
)

print(f"Gravando fisicamente na camada Bronze: {path_bronze}")

print(f"Iniciando gravação física na camada Bronze...")

# 3. CARGA FÍSICA (WRITE BRONZE) - CÓDIGO DE PRODUÇÃO
print(f"Iniciando gravação incremental na camada Bronze...")

(
    df_bronze.write
    .format("delta")
    .mode("append") # <--- Voltamos para o comportamento incremental
    .option("mergeSchema", "true") # <--- Proteção extra: se a origem enviar uma coluna nova no futuro, o Delta aceita sem quebrar
    .options(**adls_options) 
    .partitionBy("ano_particao", "mes_particao")
    .save(path_bronze)
)

print(f"SUCESSO! Ingestão incremental concluída com segurança.")

print(f"SUCESSO! O passado foi apagado e a Bronze agora tem as partições e é 100% String.")

print("Ingestão Bronze finalizada com sucesso!")
display(df_bronze.limit(5))

In [0]:

# ============================================================
# 3. CAMADA SILVER (CATEGORIAS - LIMPEZA E HIERARQUIA)
# ============================================================
from pyspark.sql.functions import current_timestamp, col, translate

print(f"Iniciando processamento da camada Silver para: {tabela}...")

# Define o caminho de destino na Silver
path_silver = f"abfss://squad3@{storage_account}.dfs.core.windows.net/silver/{tabela}"

# Lendo a tabela Delta da Bronze
df_bronze_read = (
    spark.read
    .format("delta")
    .options(**adls_options) 
    .load(path_bronze)
)

# Mapeamento para remoção de acentos (Regra 2)
com_acentos = "áàâãäéèêëíìîïóòôõöúùûüçÁÀÂÃÄÉÈÊËÍÌÎÏÓÒÔÕÖÚÙÛÜÇ"
sem_acentos = "aaaaaeeeeiiiiooooouuuucAAAAAEEEEIIIIOOOOOUUUUC"

df_silver = (
    df_bronze_read
    
    # 1. Unicidade Básica
    .dropna(subset=["id_categoria"])
    .dropDuplicates(["id_categoria"])
    
    # 2. Tipagem e Preservação de Hierarquia (O TRUQUE DO CAST DUPLO ESTÁ AQUI)
    .withColumn("id_categoria", col("id_categoria").cast("double").cast("string"))
    .withColumn("id_categoria_pai", col("id_categoria_pai").cast("double").cast("string")) 
    
    # 3. Limpeza de Caracteres Especiais
    .withColumn("nome_categoria", translate(col("nome_categoria").cast("string"), com_acentos, sem_acentos))
    
    # 4. Auditoria da camada Silver
    .withColumn("silver_processed_at", current_timestamp())
)


# ============================================================
# 4. CARGA FÍSICA (WRITE SILVER)
# ============================================================
print(f"Gravando dados limpos fisicamente em: {path_silver}")

(
    df_silver.write
    .format("delta")
    .mode("append") 
    .option("mergeSchema", "true") 
    .options(**adls_options) 
    .partitionBy("ano_particao", "mes_particao")
    .save(path_silver)
)

print("SUCESSO! Categoria refinada e sem acentos salva na Silver.\n")

display(df_silver.limit(5))

In [0]:
# ============================================================
# 5. CAMADA GOLD (KPI: DIVERSIFICAÇÃO DO CATÁLOGO)
# ============================================================
from pyspark.sql.functions import col, count, current_timestamp, year, month, date_format

print("Iniciando processamento da camada Gold (Qtd de Subcategorias por Raiz)...")

# Caminho de destino do Data Mart na Gold
tabela_gold = "kpi_subcategorias_por_raiz"
path_gold_agg = f"abfss://squad3@{storage_account}.dfs.core.windows.net/gold/{tabela_gold}"

# ------------------------------------------------------------
# TRANSFORMAÇÃO: SELF-JOIN E AGREGAÇÃO
# ------------------------------------------------------------
# 1. Filtramos quem é Categoria Raiz (não tem id_categoria_pai)
df_raiz = (
    df_silver
    .filter(col("id_categoria_pai").isNull() | (col("id_categoria_pai") == "null"))
    .select(
        col("id_categoria").alias("id_categoria_raiz"),
        col("nome_categoria").alias("nome_categoria_raiz")
    )
)

# 2. Filtramos quem é Subcategoria (tem id_categoria_pai preenchido)
df_subcategorias = (
    df_silver
    .filter(col("id_categoria_pai").isNotNull() & (col("id_categoria_pai") != "null"))
    .select(
        col("id_categoria").alias("id_subcategoria"),
        col("id_categoria_pai")
    )
)

# 3. Cruzamento e Cálculo do KPI
df_gold_agg = (
    df_subcategorias
    .join(
        df_raiz,
        df_subcategorias.id_categoria_pai == df_raiz.id_categoria_raiz,
        "inner"
    )
    .groupBy("id_categoria_raiz", "nome_categoria_raiz")
    .agg(
        # Conta quantas subcategorias pertencem a esta raiz
        count("id_subcategoria").alias("qtd_subcategorias")
    )
    
    # 4. Auditoria e Particionamento (com a máscara relacional)
    .withColumn("data_fotografia", current_timestamp())
    .withColumn("ano_particao", year(col("data_fotografia")))
    .withColumn("mes_particao", month(col("data_fotografia")))
    .withColumn("gold_processed_at", date_format(current_timestamp(), "yyyy-MM-dd HH:mm:ss"))
)

# ============================================================
# 6. CARGA FÍSICA NO DELTA LAKE (WRITE GOLD)
# ============================================================
print("Gravando Data Mart na camada Gold (MODO: APPEND)...")

(
    df_gold_agg.write
    .format("delta")
    .mode("append") # <--- Alterado para append
    .option("mergeSchema", "true") # <--- Par ideal do append para evolução segura do schema
    .options(**adls_options)
    .partitionBy("ano_particao", "mes_particao")
    .save(path_gold_agg)
)

print("SUCESSO! KPI de diversificação de categorias gravado na Gold via APPEND.\n")
display(df_gold_agg.limit(5))

In [0]:
# ============================================================
# 6. EXPORTAÇÃO PARA O SQL SERVER (SERVING LAYER SERVERLESS)
# ============================================================
import os
from dotenv import load_dotenv

print("Iniciando a exportação do KPI para o SQL Server (MODO APPEND)...")

# Garante que as variáveis do .env estão carregadas
load_dotenv(".env")

# 1. Configurações de Conexão
jdbc_hostname = os.getenv("SQL_HOST")
jdbc_port = "1433" 
jdbc_database = os.getenv("SQL_DATABASE")
jdbc_username = os.getenv("SQL_USERNAME")
jdbc_password = os.getenv("SQL_PASSWORD")

# Nome da tabela que será consumida pelo Looker/Power BI
tabela_sql_server = "dbo.KpiSubcategoriasPorRaiz" 

# 2. Gravando no SQL Server via Conector Nativo
try:
    (
        df_gold_agg.write
        .format("sqlserver")
        .option("host", jdbc_hostname)
        .option("port", jdbc_port)
        .option("database", jdbc_database)
        .option("dbtable", tabela_sql_server)
        .option("user", jdbc_username)
        .option("password", jdbc_password)
        .mode("append") 
        .save()
    )
    print(f"SUCESSO! Dados exportados perfeitamente para a tabela {tabela_sql_server} no SQL Server via APPEND.")
except Exception as e:
    print(f"Erro ao exportar para o SQL Server:\n{e}")